In [7]:
from pathlib import Path

import numpy as np
import pandas as pd


def _data_dir() -> Path:
    here = Path.cwd()
    if (here / "training_set_VU_DM.csv").is_file():
        return here
    assign2 = here / "Assignment_2"
    if (assign2 / "training_set_VU_DM.csv").is_file():
        return assign2
    raise FileNotFoundError(
        "training_set_VU_DM.csv not found; cwd should be Assignment_2 or repo root"
    )


DATA_DIR = _data_dir()
CSV_PATH = DATA_DIR / "training_set_VU_DM.csv"
USE_SAMPLE = False
SAMPLE_N = 200_000
MAKE_VAL_SET = True
FILL_NANS = True
VAL_FRAC = 0.15
FS_MAX_ROWS = 200_000
SAVE_FINAL_CSV = True
CHUNK = 500_000


def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if c == "date_time":
            continue
        if df[c].dtype == np.float64:
            df[c] = df[c].astype(np.float32)
        elif df[c].dtype == np.int64:
            lo, hi = df[c].min(), df[c].max()
            if hi <= 127 and lo >= -128:
                df[c] = df[c].astype(np.int8)
            else:
                df[c] = df[c].astype(np.int32)
    return df


def load_training_csv():
    kw = {"nrows": SAMPLE_N} if USE_SAMPLE else {}
    df = pd.read_csv(CSV_PATH, **kw)
    downcast_numeric(df)
    print(f"{'sample' if USE_SAMPLE else 'full'} -> {len(df):,} rows")
    return df


def finalized_csv_paths() -> tuple[Path, Path]:
    tag = f"sample_{SAMPLE_N}" if USE_SAMPLE else "full"
    return DATA_DIR / f"training_X_{tag}.csv", DATA_DIR / f"training_y_{tag}.csv"


def write_csv_chunked(df: pd.DataFrame, path: Path) -> None:
    for i, start in enumerate(range(0, len(df), CHUNK)):
        df.iloc[start : start + CHUNK].to_csv(
            path, mode="w" if i == 0 else "a", header=(i == 0), index=False
        )

In [8]:

stat = CSV_PATH.stat()
print(f"path: {CSV_PATH}")
print(f"size: {stat.st_size / 1e9:.3f} GB ({stat.st_size:,} bytes)")

cols = pd.read_csv(CSV_PATH, nrows=0).columns
print(f"n_columns: {len(cols)}")
print(list(cols))

path: d:\antal\DataMiningTechniques_A1\Assignment_2\training_set_VU_DM.csv
size: 1.268 GB (1,267,795,361 bytes)
n_columns: 54
['srch_id', 'date_time', 'site_id', 'visitor_location_country_id', 'visitor_hist_starrating', 'visitor_hist_adr_usd', 'prop_country_id', 'prop_id', 'prop_starrating', 'prop_review_score', 'prop_brand_bool', 'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price', 'position', 'price_usd', 'promotion_flag', 'srch_destination_id', 'srch_length_of_stay', 'srch_booking_window', 'srch_adults_count', 'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool', 'srch_query_affinity_score', 'orig_destination_distance', 'random_bool', 'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff', 'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff', 'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff', 'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff', 'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff', 'comp6_rate', 'comp6_inv', 'comp6_rate_perce

In [9]:
sample = load_training_csv()
print(sample.shape)
display(sample.head(15))

full -> 4,958,347 rows
(4958347, 54)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
5,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,30184,4,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,7.0,0,NaN,0
6,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,44147,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
7,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,50984,2,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
8,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,53341,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,6.0,0,NaN,0
9,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,56880,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


In [10]:
display(sample.dtypes.to_frame("dtype"))

numeric = sample.select_dtypes(include="number")
if numeric.shape[1]:
    display(numeric.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

na_pct = sample.isna().mean().sort_values(ascending=False) * 100
display(pd.DataFrame({"missing_pct_sample": na_pct}))

,dtype
srch_id,int32
date_time,str
site_id,int8
visitor_location_country_id,int32
visitor_hist_starrating,float32
visitor_hist_adr_usd,float32
prop_country_id,int32
prop_id,int32
prop_starrating,int8
prop_review_score,float32


,count,mean,std,min,5%,25%,50%,75%,95%,max
srch_id,4958347.0,166366.561096,96112.230102,1.000000,16605.000000,82936.000000,166507.000000,249724.000000,316004.000000,3.327850e+05
site_id,4958347.0,9.953133,7.646890,1.000000,5.000000,5.000000,5.000000,14.000000,28.000000,3.400000e+01
visitor_location_country_id,4958347.0,175.340453,65.916249,1.000000,55.000000,100.000000,219.000000,219.000000,219.000000,2.310000e+02
visitor_hist_starrating,251866.0,3.374334,0.692519,1.410000,2.150000,2.920000,3.450000,3.930000,4.500000,5.000000e+00
visitor_hist_adr_usd,252988.0,176.022659,107.254494,0.000000,65.739998,109.809998,152.240005,213.490005,356.670013,1.958700e+03
prop_country_id,4958347.0,173.973897,68.345248,1.000000,31.000000,100.000000,219.000000,219.000000,219.000000,2.300000e+02
prop_id,4958347.0,70079.179496,40609.920378,1.000000,7091.000000,35010.000000,69638.000000,105168.000000,133841.000000,1.408210e+05
prop_starrating,4958347.0,3.180525,1.051024,0.000000,2.000000,3.000000,3.000000,4.000000,5.000000,5.000000e+00
prop_review_score,4950983.0,3.777777,1.050329,0.000000,1.500000,3.500000,4.000000,4.500000,4.500000,5.000000e+00
prop_brand_bool,4958347.0,0.634699,0.481514,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000e+00


,missing_pct_sample
comp1_rate_percent_diff,98.095353
comp6_rate_percent_diff,98.060362
comp1_rate,97.581250
comp1_inv,97.387053
comp4_rate_percent_diff,97.356256
gross_bookings_usd,97.208949
comp7_rate_percent_diff,97.206428
comp6_rate,95.156511
visitor_hist_starrating,94.920364
visitor_hist_adr_usd,94.897735


## Preprocessing, feature engineering, and selection

Flags in the first cell: `USE_SAMPLE`, `MAKE_VAL_SET`, `FS_MAX_ROWS`. Feature selection uses at most `FS_MAX_ROWS` train rows.

In [11]:
import gc

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

if "load_training_csv" not in globals():
    raise RuntimeError("Run the first cell first.")

np.random.seed(0)
RNG = np.random.default_rng(0)

COMP_RATE_COLS = [f"comp{i}_rate" for i in range(1, 9)]
COMP_INV_COLS = [f"comp{i}_inv" for i in range(1, 9)]
COMP_DIFF_COLS = [f"comp{i}_rate_percent_diff" for i in range(1, 9)]

META_FOR_X = frozenset(
    {
        "srch_id",
        "prop_id",
        "relevance",
        "date_time",
        "click_bool",
        "booking_bool",
        "gross_bookings_usd",
    }
)


def build_relevance(click: pd.Series, booking: pd.Series) -> pd.Series:
    c = click.astype(bool).to_numpy()
    b = booking.astype(bool).to_numpy()
    rel = np.zeros(len(click), dtype=np.int32)
    rel[c & ~b] = 1
    rel[b] = 5
    return pd.Series(rel, index=click.index, dtype=np.int32)


def preprocess_base(df: pd.DataFrame) -> pd.DataFrame:
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    df["srch_hour"] = df["date_time"].dt.hour.astype("float32")
    df["srch_dow"] = df["date_time"].dt.dayofweek.astype("float32")
    df["srch_month"] = df["date_time"].dt.month.astype("float32")

    if FILL_NANS:
        for c in COMP_RATE_COLS + COMP_INV_COLS:
            df[c] = df[c].fillna(-999)
        for c in COMP_DIFF_COLS:
            df[c] = df[c].fillna(0.0)
        df["visitor_hist_starrating"] = df["visitor_hist_starrating"].fillna(-1.0)
        df["visitor_hist_adr_usd"] = df["visitor_hist_adr_usd"].fillna(-1.0)
        for c in [
            "prop_review_score",
            "prop_location_score2",
            "orig_destination_distance",
            "srch_query_affinity_score",
        ]:
            df[c] = df[c].fillna(df[c].median())

    return df


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df
    out["log_price_usd"] = np.log1p(out["price_usd"].clip(lower=0).to_numpy(dtype=np.float32)).astype("float32")
    out["price_minus_hist_log"] = (out["log_price_usd"] - out["prop_log_historical_price"].astype("float32")).astype(
        "float32"
    )

    vhist = out["visitor_hist_starrating"].to_numpy(dtype=np.float32)
    star = out["prop_starrating"].to_numpy(dtype=np.float32)
    star_m = star - np.where((vhist < 0) | np.isnan(vhist), np.nan, vhist)
    if FILL_NANS:
        star_m = np.nan_to_num(star_m, nan=0.0)
    out["star_minus_visitor_hist"] = star_m.astype("float32")

    out["visitor_country_eq_prop_country"] = (
        out["visitor_location_country_id"] == out["prop_country_id"]
    ).astype(np.int8)

    los = out["srch_length_of_stay"].replace(0, np.nan)
    out["price_per_night"] = (out["price_usd"] / los).astype("float32")
    if FILL_NANS:
        out["price_per_night"] = out["price_per_night"].fillna(out["price_usd"].astype("float32"))

    g = out.groupby("srch_id", sort=False)
    med_price = g["price_usd"].transform("median")
    med_safe = med_price.replace(0, np.nan)
    out["price_ratio_to_srch_median"] = (out["price_usd"] / med_safe).astype("float32")
    out["price_rank_pct_in_srch"] = g["price_usd"].rank(pct=True).astype("float32")
    out["star_rank_pct_in_srch"] = g["prop_starrating"].rank(pct=True).astype("float32")
    out["review_rank_pct_in_srch"] = g["prop_review_score"].rank(pct=True).astype("float32")
    out["dist_rank_pct_in_srch"] = g["orig_destination_distance"].rank(pct=True, ascending=True).astype(
        "float32"
    )
    out["score1_rank_pct_in_srch"] = g["prop_location_score1"].rank(pct=True).astype("float32")
    out["n_props_in_srch"] = g["prop_id"].transform("size").astype("int32")

    rates = out[COMP_RATE_COLS]
    out["comp_n_better_rate"] = (rates == 1).sum(axis=1).astype(np.int8)
    out["comp_n_worse_rate"] = (rates == -1).sum(axis=1).astype(np.int8)
    if FILL_NANS:
        out["comp_n_rate_observed"] = rates.ne(-999).sum(axis=1).astype(np.int8)
    else:
        out["comp_n_rate_observed"] = rates.notna().sum(axis=1).astype(np.int8)

    diffs = out[COMP_DIFF_COLS]
    out["comp_mean_abs_pct_diff"] = diffs.abs().mean(axis=1, skipna=True).astype("float32")

    for c in ["price_ratio_to_srch_median", "price_minus_hist_log"]:
        out[c] = out[c].replace([np.inf, -np.inf], np.nan).astype("float32")
        if FILL_NANS:
            out[c] = out[c].fillna(0).astype("float32")

    return out


def full_fe_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    preprocess_base(df)
    df.drop(columns=["date_time"], inplace=True, errors="ignore")
    add_engineered_features(df)
    df["relevance"] = build_relevance(df["click_bool"], df["booking_bool"])
    return df


def correlation_prune(df: pd.DataFrame, cols: list[str], thr: float = 0.95) -> list[str]:
    sub = df[cols].astype(np.float32)
    corr = sub.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop: set[str] = set()
    for j, c in enumerate(upper.columns):
        for r, v in upper[c].items():
            if pd.notna(v) and v > thr:
                drop.add(c)
                break
    return [c for c in cols if c not in drop]




def select_numeric_feature_columns(df: pd.DataFrame, *, include_position: bool = False) -> list[str]:
    cand = [
        c
        for c in df.columns
        if c not in META_FOR_X and pd.api.types.is_numeric_dtype(df[c])
    ]
    if not include_position:
        cand = [c for c in cand if c != "position"]
    return cand


def fit_feature_selection(
    df_train: pd.DataFrame,
    corr_thr: float = 0.95,
    max_rows: int | None = None,
) -> tuple[list[str], list[str]]:
    if max_rows is None:
        max_rows = FS_MAX_ROWS
    cand = select_numeric_feature_columns(df_train, include_position=False)
    df_fs = df_train
    if len(df_train) > max_rows:
        df_fs = df_train.sample(n=max_rows, random_state=0)
        print(f"feature selection: {max_rows:,} / {len(df_train):,} rows")
    sub = df_fs[cand].replace([np.inf, -np.inf], np.nan)
    if not FILL_NANS:
        sub = sub.fillna(0.0)
    var = sub.var(skipna=True, numeric_only=True)
    var_ok = var[var > 1e-12].index.tolist()
    pruned = correlation_prune(sub, var_ok, thr=corr_thr)
    return var_ok, pruned


fe_df = load_training_csv()
cols_before_fe = set(fe_df.columns)
full_fe_pipeline(fe_df)
gc.collect()
print(fe_df.shape, sorted(set(fe_df.columns) - cols_before_fe))

SPLIT_SEED = 0
if MAKE_VAL_SET:
    _, val_ids = train_test_split(
        fe_df["srch_id"].unique(), test_size=VAL_FRAC, random_state=SPLIT_SEED
    )
    val_ids = set(val_ids)
    fe_df["split"] = np.where(fe_df["srch_id"].isin(val_ids), "val", "train")
    n_tr = int((fe_df["split"] == "train").sum())
    n_va = int((fe_df["split"] == "val").sum())
    print(f"train {n_tr:,} rows, val {n_va:,}")
else:
    fe_df["split"] = "train"

if MAKE_VAL_SET:
    train_idx = np.flatnonzero(fe_df["split"].to_numpy() == "train")
else:
    train_idx = None

if train_idx is not None and len(train_idx) > FS_MAX_ROWS:
    fs_pick = RNG.choice(train_idx, size=FS_MAX_ROWS, replace=False)
    fs_df = fe_df.iloc[fs_pick]
else:
    fs_df = fe_df.iloc[train_idx] if train_idx is not None else fe_df

var_ok, pruned = fit_feature_selection(fs_df, corr_thr=0.95)
SELECTED_FEATURES = pruned
del fs_df
gc.collect()

mi_rows = min(30_000, len(fe_df) if train_idx is None else len(train_idx))
if train_idx is not None:
    mi_pick = RNG.choice(train_idx, size=mi_rows, replace=False) if len(train_idx) > mi_rows else train_idx
else:
    mi_pick = RNG.choice(len(fe_df), size=mi_rows, replace=False) if len(fe_df) > mi_rows else np.arange(len(fe_df))
X_mi = fe_df.iloc[mi_pick][SELECTED_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_mi = fe_df.iloc[mi_pick]["relevance"].to_numpy()
mi = mutual_info_classif(X_mi, y_mi, random_state=0, discrete_features=False)
FEATURE_MI = pd.Series(mi, index=SELECTED_FEATURES).sort_values(ascending=False)

print(len(var_ok), "after variance; dropped corr:", sorted(set(var_ok) - set(pruned)))
print(len(SELECTED_FEATURES), "selected")
print(FEATURE_MI.head(20))

keep_cols = ["srch_id", "click_bool", "booking_bool", "relevance", "split"] + SELECTED_FEATURES
fe_df = fe_df[keep_cols]
gc.collect()

y_rel = fe_df["relevance"].to_numpy()
groups = fe_df["srch_id"].to_numpy()
fe_df_ltr = fe_df[["srch_id", "relevance"] + SELECTED_FEATURES]

train_m = fe_df["split"].eq("train")
X_train = fe_df.loc[train_m, SELECTED_FEATURES]
y_train_rel = fe_df.loc[train_m, "relevance"].to_numpy()
y_train_click = fe_df.loc[train_m, "click_bool"].to_numpy()
y_train_book = fe_df.loc[train_m, "booking_bool"].to_numpy()
grp_train = fe_df.loc[train_m, "srch_id"].to_numpy()

if MAKE_VAL_SET:
    val_m = ~train_m
    X_val = fe_df.loc[val_m, SELECTED_FEATURES]
    y_val_rel = fe_df.loc[val_m, "relevance"].to_numpy()
    y_val_click = fe_df.loc[val_m, "click_bool"].to_numpy()
    y_val_book = fe_df.loc[val_m, "booking_bool"].to_numpy()
    grp_val = fe_df.loc[val_m, "srch_id"].to_numpy()
else:
    X_val = y_val_rel = y_val_click = y_val_book = grp_val = None

for split_name in (["train", "val"] if MAKE_VAL_SET else ["train"]):
    print(split_name, fe_df.loc[fe_df["split"] == split_name, "relevance"].value_counts().sort_index().to_dict())

if SAVE_FINAL_CSV:
    X_CSV_PATH, Y_CSV_PATH = finalized_csv_paths()
    feat_cols = ["srch_id", "split"] + SELECTED_FEATURES
    targ_cols = ["srch_id", "split", "click_bool", "booking_bool", "relevance"]
    write_csv_chunked(fe_df[feat_cols], X_CSV_PATH)
    write_csv_chunked(fe_df[targ_cols], Y_CSV_PATH)
    print(f"wrote {X_CSV_PATH} ({len(feat_cols)} cols)")
    print(f"wrote {Y_CSV_PATH} ({len(targ_cols)} cols), {len(fe_df):,} rows each")
else:
    X_CSV_PATH = Y_CSV_PATH = None

full -> 4,958,347 rows
(4958347, 73) ['comp_mean_abs_pct_diff', 'comp_n_better_rate', 'comp_n_rate_observed', 'comp_n_worse_rate', 'dist_rank_pct_in_srch', 'log_price_usd', 'n_props_in_srch', 'price_minus_hist_log', 'price_per_night', 'price_rank_pct_in_srch', 'price_ratio_to_srch_median', 'relevance', 'review_rank_pct_in_srch', 'score1_rank_pct_in_srch', 'srch_dow', 'srch_hour', 'srch_month', 'star_minus_visitor_hist', 'star_rank_pct_in_srch', 'visitor_country_eq_prop_country']
train 4,214,178 rows, val 744,169
66 after variance; dropped corr: ['comp1_inv', 'comp2_inv', 'comp6_inv', 'comp8_inv']
62 selected
random_bool                        0.012171
visitor_hist_starrating            0.008946
comp6_rate                         0.008246
visitor_country_eq_prop_country    0.007123
comp4_rate                         0.006148
comp3_rate                         0.006114
comp5_inv                          0.005966
comp2_rate                         0.005819
comp1_rate                      

In [12]:
# run this after CSV export, kernel still holds big data until you clear or restart
import gc

FREE_MEMORY = True
if FREE_MEMORY:
    _drop = [
        "fe_df", "X_train", "X_val", "X_mi", "fs_df", "fe_df_ltr",
        "y_rel", "groups", "y_train_rel", "y_val_rel",
        "y_train_click", "y_val_click", "y_train_book", "y_val_book",
        "grp_train", "grp_val", "train_m", "val_m", "mi_pick", "train_idx",
    ]
    for _n in _drop:
        if _n in globals():
            del globals()[_n]
    gc.collect()
    print("cleared big data objects, if pc still sluggish do kernel restart")

cleared big data objects, if pc still sluggish do kernel restart
